In [1]:
import stagpyviz as spv

# Scaling capabilities

To be able to easily work with dimensional and non-dimensional values as well as units conversion, stagpyviz comes with a class called [`Scaling`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html) that embeds the basics of the [pint](https://pint.readthedocs.io/en/stable/) package. 

For standard quantities (see list [here](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html#stagpyviz.scaling_factors) in *Available fields in the returned dictionary*) the function [`scaling_factors`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html#stagpyviz.scaling_factors) can be used to create them as:

In [2]:
scaling = spv.scaling_factors(
  Ra=1e7,
  temperature_factor=2390.0,
  temperature_unit="K",
  length_factor=2890.0e3,
  length_unit="m",
  diffusivity_factor=1.0e-6,
  diffusivity_unit="m**2/s",
  expansion_factor=1.0e-5,
  expansion_unit="1/K",
  gravity_factor=9.81,
  gravity_unit="m/s**2",
  density_factor=3300.0,
  density_unit="kg/m**3",
  viscosity_factor=1.0e22,
  viscosity_unit="Pa*s"
)

In this example we used factors that will let us convert between dimension and non-dimension values, but if one only wants to deal with dimension values a factor of ``1`` can be used.

The variable returned by the function [`scaling_factors`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html#stagpyviz.scaling_factors) is a dictionary containing instances of the [`Scaling`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html) class for each quantity.
We can print out its content like this:

In [3]:
for q in scaling:
  print(f"{scaling[q]}")

Scaling instance:
  Name: temperature
  Factor: 2390.0
  Unit: kelvin

Scaling instance:
  Name: length
  Factor: 2890000.0
  Unit: meter

Scaling instance:
  Name: diffusivity
  Factor: 1e-06
  Unit: meter ** 2 / second

Scaling instance:
  Name: expansion
  Factor: 1e-05
  Unit: 1 / kelvin

Scaling instance:
  Name: gravity
  Factor: 9.81
  Unit: meter / second ** 2

Scaling instance:
  Name: density
  Factor: 3300.0
  Unit: kilogram / meter ** 3

Scaling instance:
  Name: time
  Factor: 8.3521e+18
  Unit: second

Scaling instance:
  Name: velocity
  Factor: 3.460207612456747e-13
  Unit: meter / second

Scaling instance:
  Name: heat_source
  Factor: 2.861555776391566e-16
  Unit: kelvin / second

Scaling instance:
  Name: viscosity
  Factor: 1e+22
  Unit: pascal * second

Scaling instance:
  Name: pressure
  Factor: 1197.3036721303624
  Unit: pascal

Scaling instance:
  Name: strain_rate
  Factor: 1.1973036721303624e-19
  Unit: 1 / second

Scaling instance:
  Name: area
  Factor: 835

As you can see there already are most of the quantities we need to deal with.
However, you can always create your own.

## Creating a Scaling quantity

Creating an instance of the [`Scaling`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html) class is very simple, for this we need:

- a name
- a scaling factor
- a unit (optional)

Let's create instances for parameters that are used to compute the solidus after: 

- Hirschmann (2000). Mantle solidus: Experimental constraints and the effects of peridotite composition, *Geochemistry, Geophysics, Geosystems*, vol. 1, 2000GC000070.

This is a very good example because in the paper the parameters are not in S.I units so we need both a conversion and a non-dimensionalisation.
In the paper we are given the following:
$$
T(^{\circ}C) = aP^2 + bP + c
$$
where $a = -5.104$, $b = 132.899$, $c = 1120.661$ and $P$ is in GPa.
The left hand side is a temperature in Celsius degrees therefore the right hand side must be in the same unit. For the coefficient $a$ we need the product of $aP^2$ to be in Celsius, so it means that $a$ must be in $\left[ ^{\circ}C / GPa^2 \right]$, $b$ in $\left[ ^{\circ}C / GPa \right]$ and $c$ in $\left[ ^{\circ}C \right]$.

First, we start by creating the [`Scaling`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html) instance.
Because we already created the temperature and pressure scalings we can just re-use them, but note that you can always provide your factor and unit directly.

In [4]:
scaling["a"] = spv.Scaling(
  name="a",
  factor=scaling["temperature"].factor / scaling["pressure"].factor**2,
  unit=scaling["temperature"].unit / scaling["pressure"].unit**2
)
scaling["b"] = spv.Scaling(
  name="b",
  factor=scaling["temperature"].factor / scaling["pressure"].factor,
  unit=scaling["temperature"].unit / scaling["pressure"].unit
)
scaling["c"] = spv.Scaling(
  name="c",
  factor=scaling["temperature"].factor,
  unit=scaling["temperature"].unit
)
print(scaling["a"])
print(scaling["b"])
print(scaling["c"])

Scaling instance:
  Name: a
  Factor: 0.001667206028399
  Unit: kelvin / pascal ** 2

Scaling instance:
  Name: b
  Factor: 1.9961519
  Unit: kelvin / pascal

Scaling instance:
  Name: c
  Factor: 2390.0
  Unit: kelvin



## Converting quantities

Note that here the temperature unit is in kelvin and the pressure is in GPa therefore we first need to perform some conversion before the non-dimensionalisation. 
To do so we will make use of the method [`Scaling.to_base()`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html#stagpyviz.Scaling.to_base) to which we pass the value to be scaled and its current units:

In [5]:
a = scaling["a"].to_base(-5.104, unit="degC/GPa^2")
b = scaling["b"].to_base(132.899, unit="degC/GPa")
c = scaling["c"].to_base(1120.661, unit="degC")
print(f"a = {a:.4e} {scaling['a'].unit}")
print(f"b = {b:.4e} {scaling['b'].unit}")
print(f"c = {c:.4e} {scaling['c'].unit}")

a = -5.1040e-18 kelvin / pascal ** 2
b = 1.3290e-07 kelvin / pascal
c = 1.3938e+03 kelvin


We note that the parameters $a$ and $b$ have not been affected by the celsius to kelvin conversion while $c$ was. 
This is normal because $a$ and $b$ are variations and a variation of $1^{\circ}C$ is also a variation of 1 K while the parameter $c$ is an absolute value therefore it has been affected by the conversion between celsius and kelvin.

## Non-dimensionalisation

Now that our parameters are in their base units i.e., the one we use to define our Scaling instances (note that they do not have to be the S.I. base units we could have choosen different base units) we can safely compute their non-dimensionalisation using [`Scaling.a_dim()`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html#stagpyviz.Scaling.a_dim):

In [6]:
a = scaling["a"].a_dim(a)
b = scaling["b"].a_dim(b)
c = scaling["c"].a_dim(c)
print(f"a = {a:.4e}")
print(f"b = {b:.4e}")
print(f"c = {c:.4e}")

a = -3.0614e-15
b = 6.6578e-08
c = 5.8318e-01


## Playing with units

In addition to non-dimensionalisation we can easily play with units. 
For example, let's convert a Gyr into a Myr. 
To do so we first need to go to its base unit and then convert it to the unit we want: 

In [7]:
time = 1.0 # Gyr
time_s = scaling["time"].to_base(time, unit="Gyr")
print(f"time in seconds: {time_s} s")
time_ma = scaling["time"].to(time_s, unit="Ma")
print(f"time in million years: {time_ma} Ma")

time in seconds: 3.15576e+16 s
time in million years: 999.9999999999998 Ma


Let's check how many seconds there is in a year:

In [8]:
t = 1.0 # years
t_s = scaling["time"].to_base(t, unit="yr")
print(f"time in seconds: {t_s} s")
check = 3600.0 * 24.0 * 365.25
print(f"check:           {check} s")

time in seconds: 31557600.0 s
check:           31557600.0 s


Let's convert a cm/yr into a m/s

In [9]:
v_cma = 1.0 # cm/yr
v_m_s = scaling["velocity"].to_base(v_cma, unit="cm/yr")
print(f"velocity in m/s: {v_m_s} m/s")

velocity in m/s: 3.1688087814028947e-10 m/s
